# 1. Exploration

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import cv2
import numpy as np
import seaborn as sns
sns.set_theme(style="darkgrid")

In [ ]:
# source = "/content/gdrive/My Drive/Colab Notebooks/fer2013/fer2013.csv" 
source = 'fer2013/fer2013.csv'
data = pd.read_csv(source)
# data.head()
data['Usage'].value_counts()

In [ ]:
# categorical labels: 0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral
clases = {
    0:'angry',
    1:'disgust',
    2:'fear',
    3:'happy',
    4:'sad',
    5:'surprise',
    6:'neutral',
}

# data['emotion'].value_counts()
dict_clases ={
    clases[0]: data['emotion'].value_counts()[0],
    clases[1]: data['emotion'].value_counts()[1],
    clases[2]: data['emotion'].value_counts()[2],
    clases[3]: data['emotion'].value_counts()[3],
    clases[4]: data['emotion'].value_counts()[4],
    clases[5]: data['emotion'].value_counts()[5],
    clases[6]: data['emotion'].value_counts()[6],
}

keys = list(dict_clases.keys())
vals = [dict_clases[k] for k in keys] # get values in the same order as keys
sns.barplot(x=keys, y=vals)

In [ ]:
rows = data.shape[0]
columns = data.shape[1]
print(f'Hay {rows} filas y {columns} columnas')

In [ ]:
# from google.colab.patches import cv2_imshow

sample_imgs = []
sample_class = []

for i in range(1,10):
    id = np.random.randint(1,rows)
    
    line = data['pixels'][id] # image id
    clase = data['emotion'][id]  # image class
    
    image_data = np.asarray([int(x) for x in line.split()])  # numpy array
    img = image_data.reshape((48,48)).astype('uint8')*255  # 2-d numpy array
    img = cv2.bitwise_not(img)  # w&b bitwise inverted or do not multiply by 255
    
    # print(f'ID:{id} \n{clases[clase]}')
    # cv2_imshow(img)  # patch for colabs instead of cv2.imshow
    sample_imgs.append(cv2.flip(img, 0))
    sample_class.append(clases[clase])

In [ ]:
# !pip install seaborn-image
import seaborn_image as isns
g = isns.ImageGrid(sample_imgs, 
                   col_wrap=3, 
                   height=1.5, 
                   cmap="gray", 
                   cbar_label=sample_class, 
                   cbar=False, 
                   orientation='h')

# 2. Training

In [ ]:
import tensorflow as tf
from tensorflow.python.client import device_lib

def get_gpus():
    local_device_protos = device_lib.list_local_devices()
    return [x.name for x in local_device_protos if x.device_type == 'GPU']

print(get_gpus())
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# training

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from numpy import random
import seaborn as sns
import cv2
from tqdm import tqdm

from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K 
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, Activation, Dropout, MaxPooling2D, Flatten, Dense, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, TensorBoard, ModelCheckpoint
from tensorflow.keras import optimizers
from tensorflow.keras.models import model_from_json
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="dark")

In [ ]:
# Training  28709
# Test      7178 (public+private)
source = 'fer2013/fer2013.csv'
data = pd.read_csv(source)
data.head(10)

In [ ]:
data['emotion'].unique()

In [ ]:
# groups
# Angry, disgust, fear, sad = Upset = 0, 1, 2, 4 = 0
# Happy = Happy = 3 = 1
# Surprise = surprise = 5 = 2
# Neutral = Neutral = 6 = 3

clases_original = {
    0:'angry',
    1:'disgust',
    2:'fear',
    3:'happy',
    4:'sad',
    5:'surprise',
    6:'neutral',
}

clases_new = {
    0:'upset',
    1:'happy',
    2:'neutral',
}

data["emotion"].replace({1:0, 2:0, 4:0}, inplace=True)  # to upset
data["emotion"].replace({3:1}, inplace=True)  # to happy
data["emotion"].replace({6:2, 5:2}, inplace=True)  # to neutral

In [ ]:
data.head(10)

In [ ]:
print(data['emotion'].unique())
print(data.shape)

**adding extra dataframe**

In [ ]:
source2 = 'fer2013/fer2021.csv'
data2 = pd.read_csv(source2)
data2.head(10)

In [ ]:
frame = [data, data2]  # data2
dataFinal = pd.concat(frame)
dataFinal.shape

In [ ]:
dict_clases_new ={
    clases_new[0]: dataFinal['emotion'].value_counts()[0],
    clases_new[1]: dataFinal['emotion'].value_counts()[1],
    clases_new[2]: dataFinal['emotion'].value_counts()[2],
}

Newkeys = list(dict_clases_new.keys())
Newvals = [dict_clases_new[k] for k in Newkeys] # get values in the same order as keys
sns.barplot(x=Newkeys, y=Newvals)

In [ ]:
dataFinal['Usage'].unique()

In [ ]:
NumClasses = len(dict_clases_new)
NumClasses

In [ ]:
# splitting the dataset

KerasSplit = False
train_data = []
train_label =[]
test_data = []
test_label = []
private_data = []
private_label = []

total_train = dataFinal.shape[0]
pbar = tqdm(total=total_train, desc='Loading training images')
for index, row in dataFinal.iterrows():
    
    image_bw = np.asarray([int(x) for x in row['pixels'].split()])
    image_bw = image_bw.reshape(48, 48).astype('uint8')/255  # /255 normalized
    # image_bw = cv2.flip(image_bw, 0)
    
    if KerasSplit:
        train_data.append(image_bw)
        train_label.append(row['emotion'])
    else:
        if row['Usage'] == 'Training':
            train_data.append(image_bw)
            train_label.append(row['emotion'])
        elif row['Usage'] == 'PublicTest':
            test_data.append(image_bw)
            test_label.append(row['emotion'])
        else:
            private_data.append(image_bw)
            private_label.append(row['emotion'])
    pbar.update(1)
pbar.close()

train_data = np.array(np.expand_dims(train_data, -1))   
train_label = np.array(to_categorical(train_label, num_classes = NumClasses))
test_data = np.array(np.expand_dims(test_data, -1))
test_label = np.array(to_categorical(test_label, num_classes = NumClasses))
private_data = np.array(np.expand_dims(private_data, -1))   
private_label = np.array(to_categorical(private_label, num_classes = NumClasses))
print(f'[info training] \n training data: {len(train_data)} \n training labels: {len(train_label)}')
print(f'[public test] \n test data: {len(test_data)} \n test labels: {len(test_label)}')
print(f'[private test] \n private data: {len(private_data)} \n private labels: {len(private_label)}')

In [ ]:
id_random = random.randint(0, len(train_data))
print(train_label[id_random])
plt.imshow(train_data[id_random], cmap='gray', interpolation='nearest')
plt.axis('off')
plt.show()

In [ ]:
UniClass = len(dataFinal['emotion'].unique())
weighted = {
        0: round((1 / dataFinal['emotion'].value_counts()[0]) * (dataFinal.shape[0] / UniClass), 3),
        1: round((1 / dataFinal['emotion'].value_counts()[1]) * (dataFinal.shape[0] / UniClass), 3),
        2: round((1 / dataFinal['emotion'].value_counts()[2]) * (dataFinal.shape[0] / UniClass), 3),
}
print( dataFinal.shape[0], UniClass)
weighted

In [ ]:
print(f'im?genes para entrenamiento: {len(train_data)} \nim?genes para validaci?n: {len(test_data)}')

**Hyperparameters**

In [ ]:
epochs = 66  #66 for jup07 without weighted
batch_size = 64  # 64 for jup07 without weighted
learning_rate = 0.001  # (SGD=1E-2, Adam=1E-3)
lr_factor = 0.9  # 0.9 - learning rate factor in plateau
lr_patience = 6  # 6 - learning rate factor in plateau
validation = 0.2 # validation split
SGD = False
jup = 'jup09'

In [ ]:
data_augmentation = tf.keras.Sequential([
        layers.experimental.preprocessing.RandomFlip("horizontal", input_shape=(48, 48, 1)),
        layers.experimental.preprocessing.RandomContrast(0.2),
        layers.experimental.preprocessing.RandomZoom(0.2),
    ], name='data_augmentation')

model = Sequential(name='moodyNet')

model.add(data_augmentation)
model.add(Conv2D(64, (5, 5), activation='relu', input_shape=(48, 48, 1))) # kernel_regularizer=l2(0.01)
model.add(Conv2D(64, (5, 5), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2), strides=(2, 2)))
model.add(Dropout(0.5))
    
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
# model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
#model.add(BatchNormalization())
#model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.5))
    
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
#model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
#model.add(BatchNormalization())
#model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.5))
    
model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
#model.add(BatchNormalization())
model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
#model.add(BatchNormalization())
#model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.5))
    
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(NumClasses, activation='softmax'))

if SGD:
    optimType = optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
else:
    optimType = optimizers.Adam(learning_rate=learning_rate)

model.compile(optimizer=optimType, loss='categorical_crossentropy', metrics=['accuracy'])
    
# model.summary()

In [ ]:
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=lr_factor, patience=lr_patience)
early_stopper = EarlyStopping(monitor='val_accuracy', min_delta=0, patience=10, mode='auto')
checkpointer = ModelCheckpoint('models/'+jup+'/weights.hd5', monitor='val_loss', verbose=1, save_best_only=True)

history = model.fit(train_data,
          train_label,
          batch_size=batch_size,
          epochs=epochs, 
          callbacks=[lr_reducer, checkpointer, early_stopper], 
          # validation_split=validation,
          validation_data=(test_data, test_label),
          shuffle=True,
          # class_weight=weighted,
          verbose=1
         )

In [ ]:
model_json = model.to_json()

with open('models/'+jup+'/model.json', "w") as json_file:
    json_file.write(model_json)

model.save_weights('models/'+jup+'/model.h5')
print("Saved model to disk")

In [ ]:
metrics = ['loss', 'accuracy']  # , 'precision', 'recall'
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.figure(figsize=(16, 16))
for n, metric in enumerate(metrics):
    name = metric.replace("_", " ").capitalize()
    plt.subplot(2, 2, n+1)
    plt.plot(history.epoch, history.history[metric], color=colors[0], label='Train')
    plt.plot(history.epoch, history.history['val_'+metric], color=colors[0], linestyle="--", label='Val')
    plt.xlabel('Epoch')
    plt.ylabel(name)
    if metric == 'loss':
        plt.ylim([0, plt.ylim()[1]])
    elif metric == 'auc':
        plt.ylim([0.8,1])
    else:
        plt.ylim([0,1])

    plt.legend()
plt.savefig('models/'+jup+'/'+jup+'_metrics.jpg', bbox_inches='tight')
# plt.show()

In [ ]:
# predictions for the private dataset

private_predicted = np.argmax(model.predict(private_data), axis=1)  # otherwise still in one-hot-encode
private_labels = np.argmax(private_label, axis=1)

final_score = round(accuracy_score(private_labels, private_predicted), 4)
print (f'Accuracy score = {100*final_score}%')